In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('/home/admin/Documents/98_model/src')
sys.path.append('/data01/Documents/98_model/src')
sys.path.append('/data/98_model/src')

# Import modules

In [3]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from IPython.display import display

import seaborn as sns
import xgboost as xgb
import yaml
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import KFold

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import xgboost as xgb
from tqdm import tqdm
import re
import pickle
from sklearn.metrics import roc_curve, auc
import re

from tqdm import trange
import pickle
import os
import optuna

from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import threading
import gc

from sklearn.manifold import TSNE
import umap
import dill
from matplotlib.patches import FancyArrowPatch
from sklearn.metrics import root_mean_squared_error

from model_v4 import CascadeModel

/home/admin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/admin/.venv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [ ]:
from utils import PROCESS_STEPS, INPUT_PROFILE, DATA_TYPES
from utils import ColumnClassifier
from utils import squeeze_list

# Environment variables

In [ ]:
SHOW_DISTRIBUTION = False
PERFORM_CROSS_VALIDATION = True
PERFORM_BACKTEST = True

In [ ]:
FILEPATH_PCA_MODEL_SV = 'PCA_MODEL_SV.dill'
FILEPATH_PCA_MODEL_IQC = 'PCA_MODEL_IQC.dill'
FILEPATH_PCA_MODEL_SV_IQC = 'PCA_MODEL_SV_IQC.dill'

In [ ]:
model_name = 'N32S'
n_trials = 2000
lb_margin = 0.05
ub_margin = 0.05
threshold_constraint = 3

model_version = 'v4'

# Load data

In [ ]:
try : 
    data_path = '/home/admin/Documents/98_model/notebooks/260714_feature_engineering_qa/feature_store_v10_n32s.parquet'
    data = pd.read_parquet(data_path)
except : 
    try : 
        data_path = '/data01/Documents/98_model/notebooks/260714_feature_engineering_qa/feature_store_v10_n32s.parquet'
        data = pd.read_parquet(data_path)
    except : 
        data_path = '/data/98_model/notebooks/260714_feature_engineering_qa/feature_store_v10_n32s.parquet'
        data = pd.read_parquet(data_path)
data

In [ ]:
data_path_weekly_defect = '/home/admin/Documents/98_model/data/weekly_defect_n32s.parquet'
data_path_weekly_defect_vm2 = '/data01/Documents/98_model/data/weekly_defect_n32s.parquet'
data_path_weekly_defect_vm1 = '/data/98_model/data/weekly_defect_n32s.parquet'

try : 
    weekly_score = pd.read_parquet(data_path_weekly_defect)
except : 
    try : 
        weekly_score = pd.read_parquet(data_path_weekly_defect_vm2)
    except : 
        weekly_score = pd.read_parquet(data_path_weekly_defect_vm1)

In [ ]:
weekly_score.head().T

# Get column names

In [ ]:
column_classifier = ColumnClassifier()
df_cols = column_classifier.transform(data=data)

In [ ]:
cols_small_y = (
    df_cols.loc[lambda x :x['data_type'] == 'Small_Y', 'cols']
    .tolist()
)
cols_small_y = squeeze_list(cols_small_y)
cols_small_y = [x for x in cols_small_y if x in data.columns]
cols_small_y[:5]

In [ ]:
cols_sv = (
    df_cols.loc[lambda x :x['data_type'] == 'SV', 'cols']
    .tolist()
)
cols_sv = squeeze_list(cols_sv)
cols_sv[:5]

In [ ]:
cols_major_sv = [
    
]
# TODO : 주요 SV 받아서 변경 필요

cols_minor_sv = [

]

In [ ]:
cols_big_y = [
    'Y_NFF_A',
    'Y_NFF_E',
    'Y_NFF_J',
    'Y_NFF_F',
    'Y_NFF_R',
    'Y_NFF_V'
]

In [ ]:
cols_iqc = [x for x in data.columns if 'IQC' in x]
cols_iqc[:5]

# Preprocess data

In [ ]:
# 0. Big Y 없는 row 제거
data = data[data['Y_NFF_A'].notnull()]

In [ ]:
# 1. 중복 컬럼 제거
data = data.loc[:, ~data.columns.duplicated()]

In [ ]:
# 2. datetime 처리
col_target_date = '06_Assembly_Finished Date'
data[col_target_date] = pd.to_datetime(data[col_target_date])
data['week'] = data[col_target_date].dt.isocalendar().week
data['date'] = data[col_target_date].dt.date

In [ ]:
# 3. Category로 되어있는 Small y 인코딩
def encode_small_y(x) :
    # NG -> 불량 -> 1
    # OK -> 정상 -> 0
    if x == 'NG' :
        y = 1
    elif x == 'OK' : 
        y = 0 
    else : 
        y = x
    return y

for col in tqdm(cols_small_y) : 
    data[col] = data[col].apply(lambda x: encode_small_y(x))
    

In [ ]:
# 4. 모든 Row가 결측치인 Small y drop

drop_cols = data[cols_small_y].columns[data[cols_small_y].isna().all()]
data = data.drop(columns=drop_cols)

cols_small_y_nan_dropped = [x for x in cols_small_y if x in data.columns]

'''
# cols_small_y weekly score에 있는것만 발라내기 
cols_small_y_nan_dropped = [x for x in cols_small_y_nan_dropped if x in weekly_score.columns]

data[cols_small_y_nan_dropped].isna().sum()
'''

In [ ]:
# 5. Small y 결측치 Historical cumalative mean으로 fill
data[cols_small_y_nan_dropped].describe()


In [ ]:
hist_mean = data[cols_small_y_nan_dropped].expanding().mean().shift(1)
data[cols_small_y_nan_dropped] = data[cols_small_y_nan_dropped].fillna(hist_mean)

In [ ]:
data[cols_small_y_nan_dropped]

In [ ]:
# 6. Historical mean으로 안채워지는거 0으로 fill
data[cols_small_y_nan_dropped] = data[cols_small_y_nan_dropped].fillna(0)

In [ ]:
data[cols_small_y_nan_dropped]

In [ ]:
# 7. IQC 결측치 Historical cumalative mean으로 fill
cols_target = cols_iqc

data[cols_target].describe()

hist_mean = data[cols_target].expanding().mean().shift(1)
data[cols_target] = data[cols_target].fillna(hist_mean)

# Historical mean으로 안채워지는거 0으로 fill
data[cols_target] = data[cols_target].fillna(0)

data[cols_target]

In [ ]:
# 8. SV 결측치 Historical cumalative mean으로 fill
cols_target = cols_sv

data[cols_target].describe()

hist_mean = data[cols_target].expanding().mean().shift(1)
data[cols_target] = data[cols_target].fillna(hist_mean)

# Historical mean으로 안채워지는거 0으로 fill
data[cols_target] = data[cols_target].fillna(0)

data[cols_target]

# Show distribution

## Weekly score

In [ ]:
if model_name == 'N32S' :
    weekly_score = (
        weekly_score
        .drop('date', axis=1)
        .set_index('week')
        .loc[:, cols_big_y+[x for x in cols_small_y_nan_dropped if x in weekly_score.columns]]
    )
elif model_name == 'N33Q' : 
    weekly_score = (
        data
        .groupby('week')
        [cols_big_y + cols_small_y_nan_dropped].mean()
    )
weekly_score

In [ ]:
historical_small_y_target = (
    weekly_score
    #[[x for x in cols_small_y_nan_dropped if x in weekly_score.columns]]
    .expanding().mean()
)
historical_small_y_target

In [ ]:
historical_small_y_target = (
    data
    .loc[:, cols_small_y_nan_dropped + ['week']]
    .groupby('week')
    [cols_small_y_nan_dropped].mean()
)
historical_small_y_target

## SV distribution

In [ ]:
data_sv_count_by_week = (
    data
    .groupby('week')
    [cols_sv].count()
    #.fillna(0)
)
data_sv_count_by_week.max(axis=1)

In [ ]:
data_sv_last_by_week = (
    data
    .groupby('week')
    [cols_sv].last()
    .fillna(0)
)
data_sv_last_by_week

In [ ]:
model_path = FILEPATH_PCA_MODEL_SV
data_pca = data_sv_last_by_week

In [ ]:
if SHOW_DISTRIBUTION :
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data_pca)


    if not os.path.exists(model_path) : 
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        with open(model_path, 'wb') as f : 
            dill.dump(pca, f)
    else : 
        with open(model_path, 'rb') as f : 
            pca = dill.load(f)
        X_pca = pca.transform((X_scaled))


In [ ]:
if SHOW_DISTRIBUTION :
    print('SHOW DISTRIBUTION')
    data_pca_2dim = pd.DataFrame(
        X_pca,
        columns=['PC1', 'PC2'],
        index=getattr(data_pca, "index", None)
    )
    data_pca_2dim[cols_big_y + cols_small_y_nan_dropped] = weekly_score[cols_big_y + cols_small_y_nan_dropped]

    idx_col_y = 0
    col_y = (cols_big_y + cols_small_y_nan_dropped)[idx_col_y]

    x = data_pca_2dim['PC1'].values
    y = data_pca_2dim['PC2'].values
    score = (1-data_pca_2dim[col_y].values)
    week = data_pca_2dim.index.tolist()

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = Normalize(score.min(), score.max())

    lc = LineCollection(
        segments,
        cmap='viridis',
        norm=norm,
        linewidth=2
    )
    lc.set_array(score[:-1])

    fig, ax = plt.subplots(figsize=(7,6))
    ax.add_collection(lc)

    # Scatter
    sc = ax.scatter(
        x, y,
        c=score,
        cmap='viridis',
        norm=norm,
        s=30,
        edgecolor='k',
        zorder=3
    )

    # #1. 화살표 추가
    for i in range(len(x) - 1) : 
        arrow = FancyArrowPatch(
            (x[i], y[i]),
            (x[i+1], y[i+1]),
            arrowstyle="-|>",
            mutation_scale=10,
            color=plt.cm.viridis(norm(score[i])),
            linewidth=1.5,
            alpha=0.8,
            zorder=2,
        )
        ax.add_patch(arrow)

    # 2. 레이블 추가
    for real_idx, idx in enumerate(data_pca_2dim.index.tolist()) : 
        ax.text(
            x[real_idx],
            y[real_idx],
            f"Week : {week[real_idx]}, Score : {score[real_idx]}",
            fontsize=8,
            ha='left',
            va='bottom',
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc='white',
                alpha=0.7,
                ec='gray'
            )
        )

    ax.autoscale()

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'Defect rate {col_y}')

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Trajectory by week, by {col_y}")

    plt.tight_layout()
    plt.show()

## IQC distribution

In [ ]:
data_iqc_mean_by_week = (
    data
    .groupby('week')
    [cols_iqc].mean()
    .fillna(0)
)
data_iqc_mean_by_week

In [ ]:
model_path = FILEPATH_PCA_MODEL_IQC
data_pca = data_iqc_mean_by_week

In [ ]:
if SHOW_DISTRIBUTION :
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data_pca)


    if not os.path.exists(model_path) : 
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        with open(model_path, 'wb') as f : 
            dill.dump(pca, f)
    else : 
        with open(model_path, 'rb') as f : 
            pca = dill.load(f)
        X_pca = pca.transform((X_scaled))


In [ ]:
if SHOW_DISTRIBUTION :
    print('SHOW DISTRIBUTION')
    data_pca_2dim = pd.DataFrame(
        X_pca,
        columns=['PC1', 'PC2'],
        index=getattr(data_pca, "index", None)
    )
    data_pca_2dim[cols_big_y + cols_small_y_nan_dropped] = weekly_score[cols_big_y + cols_small_y_nan_dropped]

    idx_col_y = 0
    col_y = (cols_big_y + cols_small_y_nan_dropped)[idx_col_y]

    x = data_pca_2dim['PC1'].values
    y = data_pca_2dim['PC2'].values
    score = (1-data_pca_2dim[col_y].values)
    week = data_pca_2dim.index.tolist()

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = Normalize(score.min(), score.max())

    lc = LineCollection(
        segments,
        cmap='viridis',
        norm=norm,
        linewidth=2
    )
    lc.set_array(score[:-1])

    fig, ax = plt.subplots(figsize=(7,6))
    ax.add_collection(lc)

    # Scatter
    sc = ax.scatter(
        x, y,
        c=score,
        cmap='viridis',
        norm=norm,
        s=30,
        edgecolor='k',
        zorder=3
    )

    # #1. 화살표 추가
    for i in range(len(x) - 1) : 
        arrow = FancyArrowPatch(
            (x[i], y[i]),
            (x[i+1], y[i+1]),
            arrowstyle="-|>",
            mutation_scale=10,
            color=plt.cm.viridis(norm(score[i])),
            linewidth=1.5,
            alpha=0.8,
            zorder=2,
        )
        ax.add_patch(arrow)

    # 2. 레이블 추가
    for real_idx, idx in enumerate(data_pca_2dim.index.tolist()) : 
        ax.text(
            x[real_idx],
            y[real_idx],
            f"Week : {week[real_idx]}, Score : {score[real_idx]}",
            fontsize=8,
            ha='left',
            va='bottom',
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc='white',
                alpha=0.7,
                ec='gray'
            )
        )

    ax.autoscale()

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'Defect rate {col_y}')

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Trajectory by week, by {col_y}")

    plt.tight_layout()
    plt.show()

## SV + IQC distribution

In [ ]:
data_sv_last_iqc_mean_by_week = pd.concat([
    data_sv_last_by_week,
    data_iqc_mean_by_week
], axis=1)
data_sv_last_iqc_mean_by_week

In [ ]:
model_path = FILEPATH_PCA_MODEL_SV_IQC
data_pca = data_sv_last_iqc_mean_by_week

In [ ]:
if SHOW_DISTRIBUTION :
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data_pca)


    if not os.path.exists(model_path) : 
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        with open(model_path, 'wb') as f : 
            dill.dump(pca, f)
    else : 
        with open(model_path, 'rb') as f : 
            pca = dill.load(f)
        X_pca = pca.transform((X_scaled))


In [ ]:
if SHOW_DISTRIBUTION :
    print('SHOW DISTRIBUTION')
    data_pca_2dim = pd.DataFrame(
        X_pca,
        columns=['PC1', 'PC2'],
        index=getattr(data_pca, "index", None)
    )
    data_pca_2dim[cols_big_y + cols_small_y_nan_dropped] = weekly_score[cols_big_y + cols_small_y_nan_dropped]

    idx_col_y = 0
    col_y = (cols_big_y + cols_small_y_nan_dropped)[idx_col_y]

    x = data_pca_2dim['PC1'].values
    y = data_pca_2dim['PC2'].values
    score = (1-data_pca_2dim[col_y].values)
    week = data_pca_2dim.index.tolist()

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = Normalize(score.min(), score.max())

    lc = LineCollection(
        segments,
        cmap='viridis',
        norm=norm,
        linewidth=2
    )
    lc.set_array(score[:-1])

    fig, ax = plt.subplots(figsize=(7,6))
    ax.add_collection(lc)

    # Scatter
    sc = ax.scatter(
        x, y,
        c=score,
        cmap='viridis',
        norm=norm,
        s=30,
        edgecolor='k',
        zorder=3
    )

    # #1. 화살표 추가
    for i in range(len(x) - 1) : 
        arrow = FancyArrowPatch(
            (x[i], y[i]),
            (x[i+1], y[i+1]),
            arrowstyle="-|>",
            mutation_scale=10,
            color=plt.cm.viridis(norm(score[i])),
            linewidth=1.5,
            alpha=0.8,
            zorder=2,
        )
        ax.add_patch(arrow)

    # 2. 레이블 추가
    for real_idx, idx in enumerate(data_pca_2dim.index.tolist()) : 
        ax.text(
            x[real_idx],
            y[real_idx],
            f"Week : {week[real_idx]}, Score : {score[real_idx]}",
            fontsize=8,
            ha='left',
            va='bottom',
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc='white',
                alpha=0.7,
                ec='gray'
            )
        )

    ax.autoscale()

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'Defect rate {col_y}')

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Trajectory by week, by {col_y}")

    plt.tight_layout()
    plt.show()

# Model cross validation

In [ ]:
def clean_col_name(col):
    col=str(col)
    col=re.sub(R"_+", "_",col)
    col=re.sub(r"[^가-힣a-zA-Z0-9_]", "_", col)
    return col

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    score = []
    for week in tqdm(data_sv_last_by_week.index[:-1]) : 
        filename_result = f"results/result_cv_{model_version}_{model_name}_maximize_n_trials_{n_trials}_week_{week}.dill"
        filename_model = f"results/model_{model_version}_{model_name}_week_{week}.dill"
        if not os.path.exists(filename_result) : 
            gc.collect()
            # 1. 데이터 Split
            ## TODO : [ ] 2단계 모델 스킴으로 바꿔야 됨
            '''
            x_train = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_sv+cols_iqc]
            )
            x_test = (
                data
                .loc[lambda x : x['week']>week]
                .loc[lambda x : x['week']<=week+1]
                .loc[:, cols_sv+cols_iqc]
            )
            y_train = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_big_y+cols_small_y_nan_dropped]
            )
            y_test = (
                data
                .loc[lambda x : x['week']>week]
                .loc[lambda x : x['week']<=week+1]
                .loc[:, cols_big_y+cols_small_y_nan_dropped]
            )
            '''
            x = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_sv+cols_iqc]
            )
            y = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_big_y+cols_small_y_nan_dropped]
            )

            x_train, x_test, y_train, y_test = train_test_split(
                x,
                y,
                test_size=0.2,
                random_state=42,
                shuffle=True
            )


            x_train = x_train.astype('float32')
            y_train = y_train.astype('float32')
            x_test = x_test.astype('float32')
            y_test = y_test.astype('float32')

            print('reg = xgb.XGBRegressor')
            reg = CascadeModel(
                cols_small_y=cols_small_y_nan_dropped,
                cols_big_y=cols_big_y
            )

            reg.fit(x_train, y_train)

            y_pred_dict = reg.predict(x_test)
            y_pred_big_y = y_pred_dict['big_y']
            y_pred_small_y = y_pred_dict['small_y']



            y_pred_dict = reg.predict(x_test)
            y_pred_big_y = y_pred_dict['big_y']
            y_pred_small_y = y_pred_dict['small_y']

            # TODO : [ ] y_pred안에 cols_big_y랑 cols_small_y가 다 담겨져 나와야 함

            # TODO : [ ] metric 계산할때 따로 계산하든지 concat 처리가 필요
            y_pred = pd.concat([y_pred_big_y, y_pred_small_y], axis=1)

            metrics = []
            for idx, col in enumerate(y_test.columns) :
                y_test_by_column = y_test[col]
                y_pred_by_column = y_pred[clean_col_name(col)]

                try : 
                    roc_auc_by_column = roc_auc_score(y_test_by_column, y_pred_by_column)
                except : 
                    roc_auc_by_column = np.nan

                try : 
                    rmse_by_column =root_mean_sqaured_error(y_test_by_column, y_pred_by_column)
                except : 
                    rmse_by_column = np.nan
                metrics.append({
                    'col' : col,
                    'auroc' : roc_auc_by_column,
                    'rmse' : rmse_by_column
                })

            result = {
                'week' : week,
                'train_start' : data.loc[lambda x  :x['week']<=week][col_target_date].min(),
                'train_end' : data.loc[lambda x : x['week']<=week][col_target_date].max(),
                'test_start' : data.loc[lambda x : x['week']>week].loc[lambda x : x['week']<=week+1][col_target_date].min(),
                'test_end' : data.loc[lambda x : x['week']>week].loc[lambda x : x['week']<=week+1][col_target_date].max(),
                'metrics' : metrics,
                'artifacts' : {
                    'x_train' : x_train,
                    'x_test' : x_test,
                    'y_train' : y_train,
                    'y_test' : y_test,
                    'y_pred' : y_pred
                }
                        }
            with open(filename_result, 'wb') as f : 
                dill.dump(result, f)
            with open(filename_model, 'wb') as f : 
                dill.dump(reg, f)
        else : 
            with open(filename_result, 'rb') as f : 
                result = dill.load(f)    

        score.append(result)

# Visualize model CV result

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    result_dict = []

    for week in tqdm(data_sv_last_by_week.index[:-1]) : 
        filename_result = f"results/result_cv_{model_version}_{model_name}_maximize_n_trials_{n_trials}_week_{week}.dill"
        with open(filename_result, 'rb') as f : 
            result = dill.load(f)

        y_test = result['artifacts']['y_test']
        y_pred = pd.DataFrame(result['artifacts']['y_pred'])
        y_pred.columns = y_test.columns


        for col in y_test.columns : 
            try : 
                roc_auc = roc_auc_score(y_test[col], y_pred[col])
                rmse = np.nan
            except : 
                rmse = root_mean_squared_error(y_test[col], y_pred[col])
                roc_auc = np.nan
            result_dict.append({
                'week' : week,
                'col' : col,
                'roc_auc' : roc_auc,
                'rmse' : rmse
            })

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    score_df =(
        pd.DataFrame(result_dict)
        .pivot(
            index='week',
            columns='col',
            values=['roc_auc', 'rmse']
        )
    )
    score_df

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    filename_score = f"results/result_{model_version}_{model_name}.csv"
    score_df.to_csv(filename_score)

# Run backtest

In [ ]:
data_sv_last_by_week

In [ ]:
if PERFORM_BACKTEST : 
    print('PERFORM_BACKTEST')
    for week in tqdm(data_sv_last_by_week.index) : 
        filename_backtest_result = f"results/result_backtest_{model_version}_{model_name}_n_trials_{n_trials}_week_{week}.dill"
        filename_model = f"results/model_{model_version}_{model_name}_week_{week}.dill"
        if not os.path.exists(filename_backtest_result) : 
            gc.collect()
            # 1-1. 모델 불러오기
            print('# 1-1. 모델 불러오기')
            if os.path.exists(filename_model) :
                with open(filename_model, 'rb') as f : 
                    model = dill.load(f)

            # 2. 초기값 설정
            print('# 2. 초기값 설정')
            x0 = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_sv+cols_iqc]
                .iloc[-1]
            )

            # 3. 상/하한 설정
            print('# 3. 상/하한 설정')
            target_data = data_sv_last_by_week.loc[lambda x : x.index <= week]
            bounds = {
                col : (
                    (
                        target_data[col].min() 
                        - abs(target_data[col].min() )*lb_margin
                    ), 
                    (
                        target_data[col].max() 
                        + abs(target_data[col].max())*ub_margin
                )
                )
                for col in data_sv_last_by_week.columns
            }
            
            # 4. Constraint로 넣을 컬럼 설정
            ## 변동성 없는 컬럼 그대로 사용, IQC 그대로 사용 
            print('# 4. Constraint로 넣을 컬럼 설정')   
            cols_low_deviation = (
                (
                data_sv_last_by_week.nunique()
                [data_sv_last_by_week.nunique()<=threshold_constraint]
                .index.tolist()
                )
            )

            cols_iqc = [x for x in data.columns if 'IQC' in x]

            cols_to_fix = cols_low_deviation + cols_iqc

            # 5. 목적함수 정의
            print('# 5. 목적함수 정의')
            def objective(trial):
                gc.collect()
                print(f"Running trial {trial.number} in {threading.current_thread()}")
                # 5-1. 초기값 설정
                print('# 5-1. 초기값 설정')
                x_new = x0.copy()

                # 5-2. Constraint 제외하고 변경할 값 제안
                print('# 5-2. Constraint 제외하고 변경할 값 제안')
                opt_cols = list(set(cols_sv) - set(cols_to_fix))

                for col in opt_cols : 
                    low, high = bounds[col]
                    x_new[col] = trial.suggest_float(col, low, high)

                X = pd.DataFrame([x_new])
                #print('X')
                #display(X)

                #X.columns = [clean_col_name(x) for x in X.columns]
                #y_pred = model.predict(X)
                y_pred_dict = model.predict(X)
                y_pred_big_y = y_pred_dict['big_y']
                y_pred_small_y = y_pred_dict['small_y']

                # TODO : [ ] y_pred안에 cols_big_y랑 cols_small_y가 다 담겨져 나와야 함

                # TODO : [ ] metric 계산할때 따로 계산하든지 concat 처리가 필요
                y_pred = pd.concat([y_pred_big_y, y_pred_small_y], axis=1)
                #print('prediced y_pred')
                #display(y_pred)
                #y_pred = pd.DataFrame(y_pred)

                y_pred.columns = y_test.columns
                #print('dataframe y pred')
                #display(y_pred)

                # 전체 유형 불량에 대한 평균값 사용 (가중치 동일)
                
                # big y
                # A는 1-A 해서 minimize
                y_pred['Y_NFF_A'] = 1- y_pred['Y_NFF_A']
                mean_big_y = abs(y_pred[cols_big_y].mean().mean())
                #print('mean_big_y')
                #display(mean_big_y)
                cols_small_y_nan_dropped_result = [x for x in cols_small_y_nan_dropped if x in y_pred.columns]
                mean_small_y = np.abs((
                    (
                    y_pred[cols_small_y_nan_dropped_result].values - historical_small_y_target.loc[lambda x : x.index==week][cols_small_y_nan_dropped_result].values
                )
                )
                .mean()
                )
                #print('mean_Small_y')
                #display(mean_small_y)
                # 나머지는 그냥 minimize

                # small y 는 target과 loss 비교해서 minimize


                target = mean_big_y + mean_small_y
                return target

            # 6. 최적화
            print('# 6. 최적화')
            study = optuna.create_study(direction='minimize')
            study.optimize(objective, 
            n_trials=n_trials, 
            show_progress_bar=False, 
            n_jobs=-1
            )

            # 7. 결과 저장
            print('# 7. 결과 저장')
            pd.DataFrame([study.best_params]).to_csv(filename_backtest_result)

# Visualize backtest result

In [ ]:
with open('optuna.dill', 'wb') as f : 
    dill.dump(study, f)

In [ ]:
with open('optuna.dill', 'rb') as f : 
    study = dill.load(f)

In [ ]:
from optuna.visualization import plot_optimization_history

plot_optimization_history(study)

In [ ]:
filelist = [x for x in 
os.listdir('results')
if f"result_backtest_{model_version}_{model_name}_n_trials_{n_trials}" in x]
filelist.sort()

In [ ]:
data_sv_result = []

for filename in filelist :
    tmp = pd.read_csv(f"results/{filename}").drop('Unnamed: 0', axis=1)
    data_sv_result.append(tmp)

data_sv_result = pd.concat(data_sv_result)
data_sv_result.index = data_sv_last_by_week.index + 100

data_sv_result

In [ ]:
data_sv_mean_by_week_with_param = pd.concat([data_sv_last_by_week, data_sv_result], axis=0)
data_sv_mean_by_week_with_param = data_sv_mean_by_week_with_param
for col in data_sv_mean_by_week_with_param.columns : 
    data_sv_mean_by_week_with_param[col] = data_sv_mean_by_week_with_param[col].ffill()
data_sv_mean_by_week_with_param

In [ ]:
data_sv_mean_by_week = data_sv_mean_by_week_with_param

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_sv_mean_by_week)

filepath_pca_model = FILEPATH_PCA_MODEL_SV

if not os.path.exists(filepath_pca_model) : 
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    with open(filepath_pca_model, 'wb') as f : 
        dill.dump(pca, f)
else : 
    with open(filepath_pca_model, 'rb') as f : 
        pca = dill.load(f)
    X_pca = pca.transform((X_scaled))


In [ ]:
target_col_visualize = 'Y_NFF_A'

In [ ]:
data_sv_pca = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2'],
    index=getattr(data_sv_mean_by_week, "index", None)
)
data_sv_pca[target_col_visualize] = weekly_score[target_col_visualize]

In [ ]:
data_sv_pca

In [ ]:
x = data_sv_pca['PC1'].values
y = data_sv_pca['PC2'].values
score = (1-data_sv_pca[target_col_visualize].values)
week = data_sv_pca.index.tolist()

In [ ]:


points = np.array([x, y]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

norm = Normalize(score.min(), score.max())

lc = LineCollection(
    segments,
    cmap='viridis',
    norm=norm,
    linewidth=2
)
lc.set_array(score[:-1])

fig, ax = plt.subplots(figsize=(7,6))
ax.add_collection(lc)

# Scatter
sc = ax.scatter(
    x, y,
    c=score,
    cmap='viridis',
    norm=norm,
    s=30,
    edgecolor='k',
    zorder=3
)

# #1. 화살표 추가
for i in range(len(x) - 1) : 
    arrow = FancyArrowPatch(
        (x[i], y[i]),
        (x[i+1], y[i+1]),
        arrowstyle="-|>",
        mutation_scale=10,
        color=plt.cm.viridis(norm(score[i])),
        linewidth=1.5,
        alpha=0.8,
        zorder=2,
    )
    ax.add_patch(arrow)

# 2. 레이블 추가
for real_idx, idx in enumerate(data_sv_pca.index.tolist()) : 
    ax.text(
        x[real_idx],
        y[real_idx],
        f"Week : {week[real_idx]}, Score : {score[real_idx]}",
        fontsize=8,
        ha='left',
        va='bottom',
        bbox=dict(
            boxstyle="round,pad=0.2",
            fc='white',
            alpha=0.7,
            ec='gray'
        )
    )

ax.autoscale()

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Defect rate')

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Trajectory by date")

plt.tight_layout()
plt.show()